# Stage 3: Modelling dataset and sample definition

Stage 3 combines the age-18 outcome with the final predictor matrix at participant level using `NSID`. The predictor specification contains 69 pre-transition predictors. The domain summary contains 11 conceptual domains, of which 10 contain retained predictors.

The modelling sample is defined before data partitioning from predictor-source availability. Item-level missing values are retained. No imputation, encoding, resampling or model fitting is performed in this stage.

# Part 1: Input checks

In [1]:
# 1: Project directory definition

from pathlib import Path

working_directory = Path.cwd().resolve()

# Locate the project root when the notebook is run from either the root or a subdirectory.
project_root = next(
    (
        directory
        for directory in [working_directory, *working_directory.parents]
        if (directory / "data_derived").is_dir()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "A project directory containing data_derived was not found."
    )

data_derived = project_root / "data_derived"

print("Project root located.")
print("Data-derived directory: data_derived")

Project root located.
Data-derived directory: data_derived


In [2]:
# 2: Core input path confirmation

outcome_path = (
    data_derived
    / "stage_1_outcome_construction"
    / "age18_activity_outcome.csv"
)

predictor_path = (
    data_derived
    / "stage_2_predictor_construction"
    / "stage_2_predictor_matrix.csv"
)

print(
    "Outcome file:",
    outcome_path.relative_to(project_root).as_posix(),
)
print(f"Outcome file present: {outcome_path.is_file()}")
print(
    "Predictor file:",
    predictor_path.relative_to(project_root).as_posix(),
)
print(f"Predictor file present: {predictor_path.is_file()}")

assert outcome_path.is_file()
assert predictor_path.is_file()

Outcome file: data_derived/stage_1_outcome_construction/age18_activity_outcome.csv
Outcome file present: True
Predictor file: data_derived/stage_2_predictor_construction/stage_2_predictor_matrix.csv
Predictor file present: True


In [3]:
# 3: Outcome-file structure

import pandas as pd

# Read NSID explicitly as text so the identifier is not altered by type inference.
outcome_data = pd.read_csv(
    outcome_path,
    dtype={"NSID": "string"},
)

print(f"Shape: {outcome_data.shape}")
print("\nColumns and data types:")
print(outcome_data.dtypes.to_string())

Shape: (9767, 3)

Columns and data types:
NSID                  string
age18_outcome_code     int64
age18_outcome            str


In [4]:
# 4: Outcome identifier and value checks

print(f"Rows: {len(outcome_data):,}")
print(f"Unique NSID: {outcome_data['NSID'].nunique(dropna=True):,}")
print(f"Missing NSID: {outcome_data['NSID'].isna().sum():,}")
print(f"Duplicated NSID: {outcome_data['NSID'].duplicated().sum():,}")

print("\nMissing outcome values:")
print(
    outcome_data[
        ["age18_outcome_code", "age18_outcome"]
    ].isna().sum().to_string()
)

print("\nOutcome categories:")
print(
    outcome_data[
        ["age18_outcome_code", "age18_outcome"]
    ]
    .value_counts(sort=False)
    .sort_index()
    .to_string()
)

assert len(outcome_data) == 9_767
assert outcome_data['NSID'].notna().all()
assert outcome_data['NSID'].is_unique
assert outcome_data[['age18_outcome_code', 'age18_outcome']].notna().all().all()
assert set(outcome_data['age18_outcome_code']) == {1, 4, 5, 6}


Rows: 9,767
Unique NSID: 9,767
Missing NSID: 0
Duplicated NSID: 0

Missing outcome values:
age18_outcome_code    0
age18_outcome         0

Outcome categories:
age18_outcome_code  age18_outcome                    
1                   Education                            5132
4                   Employment                           2831
5                   Apprenticeship or training            524
6                   Unemployment or inactivity (NEET)    1280


In [5]:
# 5: Predictor-file structure

# Preserve NSID as text when the predictor matrix is loaded.
predictor_data = pd.read_csv(
    predictor_path,
    dtype={"NSID": "string"},
)

print(f"Shape: {predictor_data.shape}")
print("\nFirst 10 columns and data types:")
print(predictor_data.dtypes.head(10).to_string())

print(f"\nTotal columns: {predictor_data.shape[1]:,}")

Shape: (9767, 70)

First 10 columns and data types:
NSID                                    string
sex                                        str
birth_month_position                   float64
ethnicity                                  str
english_first_or_main_language             str
non_english_home_language              float64
highest_parental_qualification_code    float64
family_nssec_code                      float64
household_income_band                  float64
housing_tenure                         float64

Total columns: 70


In [6]:
# 6: Predictor identifier and column checks

predictor_columns = [
    column
    for column in predictor_data.columns
    if column != "NSID"
]

# Confirm that the predictor file does not contain the target columns.
outcome_columns_in_predictor_data = [
    column
    for column in predictor_data.columns
    if column in {"age18_outcome_code", "age18_outcome"}
]

print(f"Rows: {len(predictor_data):,}")
print(f"Unique NSID: {predictor_data['NSID'].nunique(dropna=True):,}")
print(f"Missing NSID: {predictor_data['NSID'].isna().sum():,}")
print(f"Duplicated NSID: {predictor_data['NSID'].duplicated().sum():,}")
print(f"Predictors: {len(predictor_columns):,}")
print(f"Duplicated column names: {predictor_data.columns.duplicated().sum():,}")
print(f"Outcome columns present: {len(outcome_columns_in_predictor_data):,}")

print("\nPredictor data types:")
print(predictor_data[predictor_columns].dtypes.value_counts().to_string())

assert len(predictor_data) == 9_767
assert predictor_data["NSID"].notna().all()
assert predictor_data["NSID"].is_unique
assert len(predictor_columns) == 69
assert not predictor_data.columns.duplicated().any()
assert not outcome_columns_in_predictor_data


Rows: 9,767
Unique NSID: 9,767
Missing NSID: 0
Duplicated NSID: 0
Predictors: 69
Duplicated column names: 0
Outcome columns present: 0

Predictor data types:
float64    65
str         4


In [7]:
# 7: Predictor-domain summary path confirmation

domain_summary_path = (
    data_derived
    / "stage_2_predictor_construction"
    / "stage_2_predictor_domain_summary.csv"
)

print(
    "Predictor-domain summary file: "
    f"{domain_summary_path.relative_to(project_root)}"
)
print(
    "Predictor-domain summary file present: "
    f"{domain_summary_path.is_file()}"
)

Predictor-domain summary file: data_derived\stage_2_predictor_construction\stage_2_predictor_domain_summary.csv
Predictor-domain summary file present: True


In [8]:
# 8: Predictor-domain summary structure

domain_summary = pd.read_csv(domain_summary_path)

print(f"Shape: {domain_summary.shape}")
print("\nColumns and data types:")
print(domain_summary.dtypes.to_string())

print("\nDomain summary:")
print(
    domain_summary[
        ["Predictor domain", "Predictors"]
    ].to_string(index=False)
)

Shape: (12, 3)

Columns and data types:
Predictor domain      str
Predictors          int64
Predictor names       str

Domain summary:
                          Predictor domain  Predictors
                    Demographic background           5
           Family socioeconomic background           7
                          Prior attainment           0
                SEN, disability and health           8
 Educational aspirations and post-16 plans           2
         School experiences and engagement           9
              Psychosocial characteristics           2
                Experiences and behaviours           7
Parental attitudes, support and engagement          16
    Post-16 social influences and guidance          10
                  School and local context           3
                                     Total          69


In [9]:
# 9: Predictor-domain correspondence

# Expand the pipe-separated predictor names so each predictor can be checked against the matrix.
domain_predictor_records = []

for _, row in domain_summary.iterrows():
    predictor_names = row["Predictor names"]

    if (
        row["Predictor domain"] == "Total"
        or pd.isna(predictor_names)
        or predictor_names == "None retained"
    ):
        continue

    for predictor_name in predictor_names.split("|"):
        domain_predictor_records.append(
            {
                "Predictor domain": row["Predictor domain"],
                "Predictor": predictor_name.strip(),
            }
        )

domain_predictor_map = pd.DataFrame(domain_predictor_records)

domain_predictor_names = domain_predictor_map["Predictor"].tolist()

duplicated_domain_predictors = sorted(
    domain_predictor_map.loc[
        domain_predictor_map["Predictor"].duplicated(keep=False),
        "Predictor",
    ].unique()
)

# Set differences identify omissions in either direction.
missing_from_domain_summary = sorted(
    set(predictor_columns) - set(domain_predictor_names)
)

not_in_predictor_matrix = sorted(
    set(domain_predictor_names) - set(predictor_columns)
)

domain_count_total = domain_summary.loc[
    domain_summary["Predictor domain"] != "Total",
    "Predictors",
].sum()

print(f"Predictors in matrix: {len(predictor_columns):,}")
print(f"Predictors listed by domain: {len(domain_predictor_names):,}")
print(f"Unique predictors listed by domain: {len(set(domain_predictor_names)):,}")
print(f"Sum of domain predictor counts: {domain_count_total:,}")
print(f"Duplicated predictors in domain summary: {len(duplicated_domain_predictors):,}")
print(f"Matrix predictors missing from domain summary: {len(missing_from_domain_summary):,}")
print(f"Domain-summary predictors absent from matrix: {len(not_in_predictor_matrix):,}")

if duplicated_domain_predictors:
    print("\nDuplicated predictors:")
    print(duplicated_domain_predictors)

if missing_from_domain_summary:
    print("\nMissing from domain summary:")
    print(missing_from_domain_summary)

if not_in_predictor_matrix:
    print("\nAbsent from predictor matrix:")
    print(not_in_predictor_matrix)

Predictors in matrix: 69
Predictors listed by domain: 69
Unique predictors listed by domain: 69
Sum of domain predictor counts: 69
Duplicated predictors in domain summary: 0
Matrix predictors missing from domain summary: 0
Domain-summary predictors absent from matrix: 0


In [10]:
# 10: Predictor counts within domains

domain_count_check = (
    domain_summary.loc[
        domain_summary["Predictor domain"].ne("Total")
    ]
    .copy()
)

domain_count_check["listed_predictors"] = (
    domain_count_check["Predictor names"]
    .fillna("")
    .apply(
        lambda names: (
            0
            if names.strip() in {"", "None retained"}
            else len(
                [
                    name
                    for name in names.split("|")
                    if name.strip()
                ]
            )
        )
    )
)

domain_count_check["counts_match"] = (
    domain_count_check["Predictors"]
    == domain_count_check["listed_predictors"]
)

total_predictors_reported = int(
    domain_summary.loc[
        domain_summary["Predictor domain"].eq("Total"),
        "Predictors",
    ].iloc[0]
)

print(
    "Domains checked: "
    f"{len(domain_count_check):,}"
)
print(
    "Domains with matching counts: "
    f"{domain_count_check['counts_match'].sum():,}"
)
print(
    "Domains with non-matching counts: "
    f"{(~domain_count_check['counts_match']).sum():,}"
)
represented_domain_count = int((domain_count_check["Predictors"] > 0).sum())

print(f"Total predictors reported: {total_predictors_reported:,}")
print(f"Conceptual domains checked: {len(domain_count_check):,}")
print(f"Domains with retained predictors: {represented_domain_count:,}")

print("\nDomain count comparison:")
print(
    domain_count_check[
        [
            "Predictor domain",
            "Predictors",
            "listed_predictors",
            "counts_match",
        ]
    ].to_string(index=False)
)

assert len(domain_count_check) == 11
assert represented_domain_count == 10
assert domain_count_check["counts_match"].all()
assert total_predictors_reported == 69
assert len(domain_predictor_names) == 69
assert len(set(domain_predictor_names)) == 69
assert not duplicated_domain_predictors
assert not missing_from_domain_summary
assert not not_in_predictor_matrix


Domains checked: 11
Domains with matching counts: 11
Domains with non-matching counts: 0
Total predictors reported: 69
Conceptual domains checked: 11
Domains with retained predictors: 10

Domain count comparison:
                          Predictor domain  Predictors  listed_predictors  counts_match
                    Demographic background           5                  5          True
           Family socioeconomic background           7                  7          True
                          Prior attainment           0                  0          True
                SEN, disability and health           8                  8          True
 Educational aspirations and post-16 plans           2                  2          True
         School experiences and engagement           9                  9          True
              Psychosocial characteristics           2                  2          True
                Experiences and behaviours           7                  7          

## Predictor-domain check

Each retained predictor must appear exactly once in the domain summary and must be present in the predictor matrix. Prior attainment remains a conceptual domain with no retained predictor.

## Stage 3 scope

Predictor timing, source-variable selection and domain assignment are not changed here. The final predictor output is checked, merged with the outcome, and used to define the modelling sample. Participant eligibility is based on predictor-source coverage.

# Part 2: Identifier comparison and dataset merge

In [11]:
# 1: Identifier comparison

outcome_ids = set(outcome_data["NSID"])
predictor_ids = set(predictor_data["NSID"])

# Set operations check whether either file contains unmatched participant identifiers.
common_ids = outcome_ids & predictor_ids
outcome_only_ids = outcome_ids - predictor_ids
predictor_only_ids = predictor_ids - outcome_ids

print(f"Outcome NSID: {len(outcome_ids):,}")
print(f"Predictor NSID: {len(predictor_ids):,}")
print(f"Common NSID: {len(common_ids):,}")
print(f"Outcome only: {len(outcome_only_ids):,}")
print(f"Predictor only: {len(predictor_only_ids):,}")

assert len(common_ids) == 9_767
assert not outcome_only_ids
assert not predictor_only_ids


Outcome NSID: 9,767
Predictor NSID: 9,767
Common NSID: 9,767
Outcome only: 0
Predictor only: 0


In [12]:
# 2: One-to-one dataset merge

# validate='one_to_one' raises an error if either input contains duplicate participant identifiers.
# indicator=True records the source match for every merged row.
modelling_data = outcome_data.merge(
    predictor_data,
    on="NSID",
    how="inner",
    validate="one_to_one",
    indicator=True,
)

print(f"Merged shape: {modelling_data.shape}")
print("\nMerge result:")
print(modelling_data["_merge"].value_counts().to_string())

assert len(modelling_data) == 9_767
assert modelling_data['_merge'].eq('both').all()


Merged shape: (9767, 73)

Merge result:
_merge
both          9767
left_only        0
right_only       0


In [13]:
# 3: Value preservation checks

# Sort by NSID before comparison so input row order does not affect the preservation check.
outcome_preserved = (
    modelling_data[outcome_data.columns]
    .sort_values("NSID")
    .reset_index(drop=True)
    .equals(
        outcome_data
        .sort_values("NSID")
        .reset_index(drop=True)
    )
)

predictors_preserved = (
    modelling_data[predictor_data.columns]
    .sort_values("NSID")
    .reset_index(drop=True)
    .equals(
        predictor_data
        .sort_values("NSID")
        .reset_index(drop=True)
    )
)

print(f"Outcome values preserved: {outcome_preserved}")
print(f"Predictor values preserved: {predictors_preserved}")

assert outcome_preserved
assert predictors_preserved


Outcome values preserved: True
Predictor values preserved: True


In [14]:
# 4: Merged dataset structure

if "_merge" in modelling_data.columns:
    modelling_data = modelling_data.drop(columns="_merge")

expected_columns = (
    ["NSID", "age18_outcome_code", "age18_outcome"]
    + predictor_columns
)

missing_columns = [
    column
    for column in expected_columns
    if column not in modelling_data.columns
]

unexpected_columns = [
    column
    for column in modelling_data.columns
    if column not in expected_columns
]

print(f"Shape: {modelling_data.shape}")
print(f"Unique NSID: {modelling_data['NSID'].nunique(dropna=True):,}")
print(f"Duplicated NSID: {modelling_data['NSID'].duplicated().sum():,}")
print(f"Expected columns: {len(expected_columns):,}")
print(f"Missing expected columns: {len(missing_columns):,}")
print(f"Unexpected columns: {len(unexpected_columns):,}")
print(
    "Expected column order: "
    f"{modelling_data.columns.tolist() == expected_columns}"
)

assert modelling_data.shape == (9_767, 72)
assert modelling_data['NSID'].is_unique
assert not missing_columns
assert not unexpected_columns
assert modelling_data.columns.tolist() == expected_columns


Shape: (9767, 72)
Unique NSID: 9,767
Duplicated NSID: 0
Expected columns: 72
Missing expected columns: 0
Unexpected columns: 0
Expected column order: True


# Part 3: Predictor availability and modelling sample definition

Participant-level predictor coverage is examined to distinguish item-level missingness from structural absence of the main early-wave predictor history. Eligibility is defined from the Wave 1 source-record indicator before outcome composition is examined.

In [15]:
# 1: Participant-level predictor availability

# Count missing values across predictor columns for each participant; the outcome is not included.
predictor_missing_count = modelling_data[predictor_columns].isna().sum(axis=1)
predictor_observed_count = len(predictor_columns) - predictor_missing_count

print(f"Participants: {len(modelling_data):,}")
print(f"Predictors per participant: {len(predictor_columns)}")
print(f"Participants with no missing predictors: {(predictor_missing_count == 0).sum():,}")
print(f"Participants with at least one missing predictor: {(predictor_missing_count > 0).sum():,}")
print(f"Participants with no observed predictors: {(predictor_observed_count == 0).sum():,}")

print("\nObserved predictors per participant:")
print(
    predictor_observed_count
    .describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
    .to_string()
)

Participants: 9,767
Predictors per participant: 69
Participants with no missing predictors: 5,203
Participants with at least one missing predictor: 4,564
Participants with no observed predictors: 0

Observed predictors per participant:
count    9767.000000
mean       66.184089
std        10.284948
min         1.000000
1%          4.000000
5%         61.000000
25%        68.000000
50%        69.000000
75%        69.000000
95%        69.000000
99%        69.000000
max        69.000000


## Participant-level predictor availability

Observed predictor counts are recalculated from the 69-predictor matrix. These counts describe coverage and are not used directly as the eligibility rule.

In [16]:
# 2: Distribution of limited predictor availability

availability_distribution = (
    predictor_observed_count
    .value_counts()
    .sort_index()
    .rename_axis("observed_predictors")
    .reset_index(name="participants")
)

availability_distribution["missing_predictors"] = (
    len(predictor_columns)
    - availability_distribution["observed_predictors"]
)

availability_distribution["percentage"] = (
    availability_distribution["participants"]
    .div(len(modelling_data))
    .mul(100)
    .round(2)
)

five_or_fewer_distribution = availability_distribution.loc[
    availability_distribution["observed_predictors"].le(5)
].copy()

counts_above_five = availability_distribution.loc[
    availability_distribution["observed_predictors"].gt(5),
    "observed_predictors",
]

next_observed_count = (
    int(counts_above_five.min())
    if not counts_above_five.empty
    else None
)

print("Observed-predictor counts at five or fewer:")
print(five_or_fewer_distribution.to_string(index=False))

print(
    "\nParticipants with five or fewer predictors: "
    f"{five_or_fewer_distribution['participants'].sum():,}"
)
print(
    "Next observed-predictor count above five: "
    f"{next_observed_count}"
)


Observed-predictor counts at five or fewer:
 observed_predictors  participants  missing_predictors  percentage
                   1             1                  68        0.01
                   2            38                  67        0.39
                   3             7                  66        0.07
                   4           197                  65        2.02

Participants with five or fewer predictors: 243
Next observed-predictor count above five: 29


## Limited predictor availability

A group of 243 participants has only one to four observed predictors; the next observed-predictor count is 29. The sample-support file is used to determine whether this low coverage reflects source-record absence.

In [17]:
# 3: Stage 2 sample-support structure

predictor_support_path = (
    data_derived
    / "stage_2_predictor_construction"
    / "stage_2_predictor_sample_support.csv"
)

predictor_support = pd.read_csv(
    predictor_support_path,
    dtype={"NSID": "string"},
)

print(f"Shape: {predictor_support.shape}")
print("\nColumns and data types:")
print(predictor_support.dtypes.to_string())

Shape: (9767, 4)

Columns and data types:
NSID                                  string
wave_1_source_record_present            bool
predictors_available                   int64
five_or_fewer_predictors_available      bool


In [18]:
# 4: Sample-support alignment

observed_counts = modelling_data[["NSID"]].copy()
observed_counts["observed_predictors"] = predictor_observed_count

# Align recalculated coverage with the saved support file by participant identifier.
support_check = observed_counts.merge(
    predictor_support,
    on="NSID",
    how="left",
    validate="one_to_one",
)

predictor_count_matches = (
    support_check["predictors_available"]
    .eq(support_check["observed_predictors"])
)

missing_support_records = (
    support_check["predictors_available"]
    .isna()
)

non_matching_counts = (
    ~predictor_count_matches
    & ~missing_support_records
)

print(f"Rows: {len(support_check):,}")
print(f"Unique NSID: {support_check['NSID'].nunique(dropna=True):,}")
print(f"Missing support records: {missing_support_records.sum():,}")
print(f"Matching predictor counts: {predictor_count_matches.sum():,}")
print(f"Non-matching predictor counts: {non_matching_counts.sum():,}")

assert len(support_check) == 9_767
assert not missing_support_records.any()
assert not non_matching_counts.any()


Rows: 9,767
Unique NSID: 9,767
Missing support records: 0
Matching predictor counts: 9,767
Non-matching predictor counts: 0


In [19]:
# 5: Source-record support for participants with five or fewer predictors

# observed=True reports only combinations that occur in the data.
support_summary = (
    predictor_support
    .groupby(
        ["five_or_fewer_predictors_available", "wave_1_source_record_present"],
        observed=True
    )
    .size()
    .rename("participants")
    .reset_index()
)

print(support_summary.to_string(index=False))

 five_or_fewer_predictors_available  wave_1_source_record_present  participants
                              False                          True          9524
                               True                         False           243


## Wave 1 source-record support

The low-coverage indicator is cross-tabulated with the Wave 1 source-record flag. Eligibility is defined from the source-record flag rather than from an observed-predictor threshold.

In [20]:
# 6: Predictor availability without a Wave 1 source record

# Define the group using Stage 2 source-record status.
no_wave_1_ids = predictor_support.loc[
    ~predictor_support["wave_1_source_record_present"],
    "NSID",
]

no_wave_1_data = modelling_data[
    modelling_data["NSID"].isin(no_wave_1_ids)
]

# Count non-missing observations predictor by predictor within this source-record group.
available_predictors = (
    no_wave_1_data[predictor_columns]
    .notna()
    .sum()
    .loc[lambda counts: counts > 0]
    .sort_values(ascending=False)
    .rename_axis("predictor")
    .reset_index(name="participants")
)

available_predictors["percentage"] = (
    available_predictors["participants"]
    / len(no_wave_1_data)
    * 100
).round(2)

print(
    "Participants without a Wave 1 source record: "
    f"{len(no_wave_1_data):,}"
)
print(
    "Predictors with any observed values: "
    f"{len(available_predictors):,}"
)
print(
    "Predictors missing for all participants: "
    f"{len(predictor_columns) - len(available_predictors):,}"
)

print("\nAvailable predictor support:")
print(available_predictors.to_string(index=False))

assert len(no_wave_1_data) == 243
assert len(available_predictors) == 4


Participants without a Wave 1 source record: 243
Predictors with any observed values: 4
Predictors missing for all participants: 65

Available predictor support:
                               predictor  participants  percentage
                               ethnicity           241       99.18
                                     sex           237       97.53
                sen_status_pretransition           204       83.95
current_statement_of_needs_pretransition           204       83.95


In [21]:
# 7: Domain coverage of predictors missing without a Wave 1 source record

missing_predictors_no_wave_1 = (
    no_wave_1_data[predictor_columns]
    .notna()
    .sum()
    .loc[lambda counts: counts == 0]
    .index
    .tolist()
)

# Map completely unavailable predictors back to the retained domain specification.
missing_predictor_domains = domain_predictor_map[
    domain_predictor_map["Predictor"].isin(
        missing_predictors_no_wave_1
    )
]

domain_coverage = (
    missing_predictor_domains
    .groupby("Predictor domain")
    .size()
    .rename("predictors")
    .reset_index()
    .sort_values("Predictor domain")
)

unmapped_predictors = sorted(
    set(missing_predictors_no_wave_1)
    - set(missing_predictor_domains["Predictor"])
)

print(
    "Predictors missing for all 243 participants: "
    f"{len(missing_predictors_no_wave_1):,}"
)
print(
    "Predictors matched to a domain: "
    f"{len(missing_predictor_domains):,}"
)
print(
    "Domains represented: "
    f"{domain_coverage['Predictor domain'].nunique():,}"
)
print(f"Unmapped predictors: {len(unmapped_predictors):,}")

print("\nDomain coverage:")
print(domain_coverage.to_string(index=False))

assert len(missing_predictors_no_wave_1) == 65
assert len(unmapped_predictors) == 0
assert domain_coverage['Predictor domain'].nunique() == 10


Predictors missing for all 243 participants: 65
Predictors matched to a domain: 65
Domains represented: 10
Unmapped predictors: 0

Domain coverage:
                          Predictor domain  predictors
                    Demographic background           3
 Educational aspirations and post-16 plans           2
                Experiences and behaviours           7
           Family socioeconomic background           7
Parental attitudes, support and engagement          16
    Post-16 social influences and guidance          10
              Psychosocial characteristics           2
                SEN, disability and health           6
                  School and local context           3
         School experiences and engagement           9


## Domain coverage without a Wave 1 source record

Predictors unavailable to all participants without a Wave 1 source record are mapped to their domains to assess how broadly the structural absence extends across the predictor specification.

In [22]:
# 8: Modelling sample definition

# Define eligibility from the source-record indicator rather than from observed-predictor counts.
eligible_ids = set(
    predictor_support.loc[
        predictor_support["wave_1_source_record_present"],
        "NSID",
    ]
)

modelling_sample = (
    modelling_data[
        modelling_data["NSID"].isin(eligible_ids)
    ]
    .copy()
    .reset_index(drop=True)
)

excluded_sample = (
    modelling_data[
        ~modelling_data["NSID"].isin(eligible_ids)
    ]
    .copy()
    .reset_index(drop=True)
)

included_predictor_counts = (
    modelling_sample[predictor_columns]
    .notna()
    .sum(axis=1)
)

excluded_predictor_counts = (
    excluded_sample[predictor_columns]
    .notna()
    .sum(axis=1)
)

overlapping_ids = (
    set(modelling_sample["NSID"])
    & set(excluded_sample["NSID"])
)

print(f"Matched participants: {len(modelling_data):,}")
print(f"Included participants: {len(modelling_sample):,}")
print(f"Excluded participants: {len(excluded_sample):,}")
print(
    "Percentage included: "
    f"{len(modelling_sample) / len(modelling_data) * 100:.2f}%"
)

print(f"\nOverlapping identifiers: {len(overlapping_ids):,}")
print(
    "All participants accounted for: "
    f"{len(modelling_sample) + len(excluded_sample) == len(modelling_data)}"
)

print(
    "\nObserved predictors among included participants: "
    f"{included_predictor_counts.min()} to "
    f"{included_predictor_counts.max()}"
)
print(
    "Observed predictors among excluded participants: "
    f"{excluded_predictor_counts.min()} to "
    f"{excluded_predictor_counts.max()}"
)

assert len(modelling_data) == 9_767
assert len(modelling_sample) == 9_524
assert len(excluded_sample) == 243
assert not overlapping_ids
assert included_predictor_counts.min() > 5
assert excluded_predictor_counts.max() <= 5


Matched participants: 9,767
Included participants: 9,524
Excluded participants: 243
Percentage included: 97.51%

Overlapping identifiers: 0
All participants accounted for: True

Observed predictors among included participants: 29 to 69
Observed predictors among excluded participants: 1 to 4


## Modelling sample definition

The modelling sample is restricted to participants with a Wave 1 source record. This is a source-availability rule rather than a complete-case rule.

Participants retained in the modelling sample may still have missing values for individual predictors. These values remain unchanged for later preprocessing after data partitioning.

In [23]:
# 9: Outcome composition by modelling sample status

# Summarise the outcome only after participant eligibility has been fixed.
outcome_group_columns = [
    "age18_outcome_code",
    "age18_outcome",
]

full_outcome_counts = (
    modelling_data
    .groupby(outcome_group_columns)
    .size()
    .rename("matched_participants")
)

included_outcome_counts = (
    modelling_sample
    .groupby(outcome_group_columns)
    .size()
    .rename("included_participants")
)

excluded_outcome_counts = (
    excluded_sample
    .groupby(outcome_group_columns)
    .size()
    .rename("excluded_participants")
)

outcome_sample_accounting = (
    pd.concat(
        [
            full_outcome_counts,
            included_outcome_counts,
            excluded_outcome_counts,
        ],
        axis=1,
    )
    .fillna(0)
    .astype(int)
    .reset_index()
)

outcome_sample_accounting["matched_percentage"] = (
    outcome_sample_accounting["matched_participants"]
    / len(modelling_data)
    * 100
).round(2)

outcome_sample_accounting["included_percentage"] = (
    outcome_sample_accounting["included_participants"]
    / len(modelling_sample)
    * 100
).round(2)

outcome_sample_accounting["excluded_percentage"] = (
    outcome_sample_accounting["excluded_participants"]
    / len(excluded_sample)
    * 100
).round(2)

outcome_sample_accounting["percentage_excluded_within_outcome"] = (
    outcome_sample_accounting["excluded_participants"]
    / outcome_sample_accounting["matched_participants"]
    * 100
).round(2)

print("Outcome composition by modelling sample status:")
print(outcome_sample_accounting.to_string(index=False))

assert outcome_sample_accounting['included_participants'].sum() == len(modelling_sample)
assert outcome_sample_accounting['excluded_participants'].sum() == len(excluded_sample)
assert outcome_sample_accounting['age18_outcome_code'].nunique() == 4


Outcome composition by modelling sample status:
 age18_outcome_code                     age18_outcome  matched_participants  included_participants  excluded_participants  matched_percentage  included_percentage  excluded_percentage  percentage_excluded_within_outcome
                  1                         Education                  5132                   4952                    180               52.54                51.99                74.07                                3.51
                  4                        Employment                  2831                   2801                     30               28.99                29.41                12.35                                1.06
                  5        Apprenticeship or training                   524                    520                      4                5.37                 5.46                 1.65                                0.76
                  6 Unemployment or inactivity (NEET)                  1

## Outcome composition after sample definition

Outcome composition is examined after the eligibility rule has been applied. The check confirms that all four outcome categories remain represented.

In [24]:
# 10: Sample accounting

sample_accounting = pd.DataFrame(
    {
        "Stage": [
            "Matched outcome and predictor sample",
            "Excluded without a Wave 1 source record",
            "Final modelling sample",
        ],
        "Participants": [
            len(modelling_data),
            len(excluded_sample),
            len(modelling_sample),
        ],
    }
)

print(sample_accounting.to_string(index=False))

                                  Stage  Participants
   Matched outcome and predictor sample          9767
Excluded without a Wave 1 source record           243
                 Final modelling sample          9524


# Part 4: Output definition and verification

In [25]:
# 1: Stage 3 output paths

output_directory = (
    data_derived
    / "stage_3_final_modelling_dataset"
)

matched_dataset_path = (
    output_directory
    / "matched_modelling_dataset.csv"
)

final_dataset_path = (
    output_directory
    / "final_modelling_dataset.csv"
)

sample_accounting_path = (
    output_directory
    / "sample_accounting.csv"
)

output_paths = {
    "Matched dataset": matched_dataset_path,
    "Final modelling dataset": final_dataset_path,
    "Sample accounting": sample_accounting_path,
}

print(
    "Output directory:",
    output_directory.relative_to(project_root).as_posix(),
)

for label, path in output_paths.items():
    print(
        f"{label}: "
        f"{path.relative_to(project_root).as_posix()}"
    )

Output directory: data_derived/stage_3_final_modelling_dataset
Matched dataset: data_derived/stage_3_final_modelling_dataset/matched_modelling_dataset.csv
Final modelling dataset: data_derived/stage_3_final_modelling_dataset/final_modelling_dataset.csv
Sample accounting: data_derived/stage_3_final_modelling_dataset/sample_accounting.csv


In [26]:
# 2: Stage 3 output saving

output_directory.mkdir(
    parents=True,
    exist_ok=True,
)

outputs = [
    (
        "Matched dataset",
        modelling_data,
        matched_dataset_path,
    ),
    (
        "Final modelling dataset",
        modelling_sample,
        final_dataset_path,
    ),
    (
        "Sample accounting",
        sample_accounting,
        sample_accounting_path,
    ),
]

for label, data, path in outputs:
    data.to_csv(
        path,
        index=False,
    )
    print(
        f"{label} saved: {path.is_file()}"
    )


Matched dataset saved: True
Final modelling dataset saved: True
Sample accounting saved: True


In [27]:
# 3: Stage 3 output round-trip verification

saved_matched_data = pd.read_csv(
    matched_dataset_path,
    dtype={"NSID": "string"},
)

saved_final_data = pd.read_csv(
    final_dataset_path,
    dtype={"NSID": "string"},
)

saved_sample_accounting = pd.read_csv(
    sample_accounting_path,
)

expected_matched_data = modelling_data.reset_index(drop=True).copy()
expected_final_data = modelling_sample.reset_index(drop=True).copy()
expected_sample_accounting = sample_accounting.reset_index(drop=True).copy()

expected_matched_data["NSID"] = (
    expected_matched_data["NSID"].astype("string")
)
expected_final_data["NSID"] = (
    expected_final_data["NSID"].astype("string")
)

# Compare saved CSV content with the in-memory data. Dtype differences caused by CSV reload are ignored.
pd.testing.assert_frame_equal(
    saved_matched_data,
    expected_matched_data,
    check_dtype=False,
    check_like=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)

# Floating-point values are compared with a tight tolerance after the CSV round trip.
pd.testing.assert_frame_equal(
    saved_final_data,
    expected_final_data,
    check_dtype=False,
    check_like=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)

pd.testing.assert_frame_equal(
    saved_sample_accounting,
    expected_sample_accounting,
    check_dtype=False,
    check_like=False,
)

print("Matched dataset round-trip verified: True")
print(f"Matched dataset shape: {saved_matched_data.shape}")
print(
    "Matched dataset duplicated NSID: "
    f"{saved_matched_data['NSID'].duplicated().sum():,}"
)

print("\nFinal modelling dataset round-trip verified: True")
print(f"Final modelling dataset shape: {saved_final_data.shape}")
print(
    "Final modelling dataset duplicated NSID: "
    f"{saved_final_data['NSID'].duplicated().sum():,}"
)
print(
    "Final modelling outcome categories: "
    f"{saved_final_data['age18_outcome_code'].nunique():,}"
)
print(
    "Final modelling predictors: "
    f"{len(predictor_columns):,}"
)

print("\nSample accounting round-trip verified: True")
print(saved_sample_accounting.to_string(index=False))

Matched dataset round-trip verified: True
Matched dataset shape: (9767, 72)
Matched dataset duplicated NSID: 0

Final modelling dataset round-trip verified: True
Final modelling dataset shape: (9524, 72)
Final modelling dataset duplicated NSID: 0
Final modelling outcome categories: 4
Final modelling predictors: 69

Sample accounting round-trip verified: True
                                  Stage  Participants
   Matched outcome and predictor sample          9767
Excluded without a Wave 1 source record           243
                 Final modelling sample          9524


## Stage 3 summary

All 9,767 outcome records match one-to-one with the predictor records. The merged dataset contains `NSID`, two outcome columns and 69 predictors.

The 243 participants without a Wave 1 source record are excluded because most predictors are structurally unavailable for this group. The final modelling sample contains 9,524 participants; item-level missing predictor values are retained.

The matched dataset, final modelling dataset and sample accounting are saved and checked by reading the files back.